In [3]:
import re
import requests

from bs4 import BeautifulSoup

In [4]:
# Scrape a player page
url = "https://www.transfermarkt.us/erick-noriega/profil/spieler/723853"
player_id = url.split('/')[-1]

In [5]:
# Get headers from https://www.whatismybrowser.com/detect/what-http-headers-is-my-browser-sending
headers = {
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36',
}

In [6]:
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")

In [7]:
response.status_code

200

In [8]:
player_name = soup.select_one('h1[class="data-header__headline-wrapper"]').text.split('\n')[-1].strip()
player_number = soup.select_one('span[class="data-header__shirt-number"]').text.strip().replace('#', '')

In [21]:
player_name, player_number

('Erick Noriega', '4')

In [14]:
# Let's use regex to find the player's contract expiry date
contract_expiry_match = re.search("Contract expires: (.*)", soup.text)
player_contract_expiry = contract_expiry_match.group(1) if contract_expiry_match else None

# Now let's add some more complex regex patterns to find the player's birthplace
birthplace_match = re.search("Place of birth:.*?([A-z].*?) ", soup.text, re.DOTALL)
player_birthplace = birthplace_match.group(1).strip() if birthplace_match else None

# We can reuse both of those ideas to find other player information but just replacing the name

# We might have to switch a couple of things though in the regex pattern like accounting for two words
agent_match = re.search("Agent:.*?([A-z].*?)\n", soup.text, re.DOTALL)
player_agent = agent_match.group(1).strip() if agent_match else None

# We can also use regex to find the player's height which has numbers a comma and letters
height_match = re.search("Height:.*?([0-9].*?)\n", soup.text, re.DOTALL)
player_height = height_match.group(1).strip() if height_match else None

In [20]:
player_contract_expiry, player_birthplace, player_agent, player_height

('Dec 31, 2027', 'Nagoya', None, '1,85 m')

In [ ]:
# Ok so now that we've got some basic regex patterns, let's move on to using api endpoints
# curlconverter.com
api_endpoints = [
    f"marketValueDevelopment/graph/{player_id}",
    f"transferHistory/list/{player_id}",
    f"player/{player_id}/performance"
]

for endpoint in api_endpoints:
    response = requests.get(
        'https://www.transfermarkt.us/ceapi/' + endpoint,
        headers=headers
    ).json()
    
    print(f"Endpoint: {endpoint}")
    print(response)
    print("\n")

Endpoint: marketValueDevelopment/graph/723853
{'list': [{'x': 1611702000000, 'y': 50000, 'mw': '€50k', 'datum_mw': 'Jan 27, 2021', 'verein': 'Machida Zelvia', 'age': '19', 'wappen': 'https://tmssl.akamaized.net//images/wappen/profil/23568.png?lm=1687778899'}, {'x': 1624917600000, 'y': 50000, 'mw': '€50k', 'datum_mw': 'Jun 29, 2021', 'verein': 'Shimizu S-Pulse', 'age': '19', 'wappen': 'https://tmssl.akamaized.net//images/wappen/profil/1062.png?lm=1575994292'}, {'x': 1642114800000, 'y': 50000, 'mw': '€50k', 'datum_mw': 'Jan 14, 2022', 'verein': 'Shimizu S-Pulse', 'age': '20', 'wappen': ''}, {'x': 1685656800000, 'y': 75000, 'mw': '€75k', 'datum_mw': 'Jun 2, 2023', 'verein': 'Universidad San Martín de Porres', 'age': '21', 'wappen': 'https://tmssl.akamaized.net//images/wappen/profil/11622.png?lm=1688505994'}, {'x': 1700002800000, 'y': 225000, 'mw': '€225k', 'datum_mw': 'Nov 15, 2023', 'verein': 'Universidad San Martín de Porres', 'age': '22', 'wappen': ''}, {'x': 1716847200000, 'y': 400000